In [2]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tensorflow.keras.models import load_model

# 1. Muat Model LSTM yang Sudah Dilatih
MODEL_KERAS_PATH = 'model_lstm_bisindo.h5'
if not os.path.exists(MODEL_KERAS_PATH):
    print(f"Error: {MODEL_KERAS_PATH} tidak ditemukan. Simpan model terlebih dahulu.")
    exit()

print("Memuat model LSTM...")
model = load_model(MODEL_KERAS_PATH)

# 2. Muat Pemetaan Label
label_map_rev = {}
if os.path.exists('label_map.txt'):
    with open('label_map.txt', 'r') as f:
        for line in f:
            parts = line.strip().split(':')
            if len(parts) == 2:
                cls_name, idx = parts[0], int(parts[1])
                label_map_rev[idx] = cls_name

# 3. Inisialisasi MediaPipe Tasks API
MODEL_PATH = 'hand_landmarker.task'
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3
)
detector = vision.HandLandmarker.create_from_options(options)

# Definisi manual koneksi tulang jari untuk menggambar garis
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Ibu jari
    (0, 5), (5, 6), (6, 7), (7, 8),         # Telunjuk
    (5, 9), (9, 10), (10, 11), (11, 12),    # Tengah
    (9, 13), (13, 14), (14, 15), (15, 16),  # Manis
    (13, 17), (17, 18), (18, 19), (19, 20), # Kelingking
    (0, 17)                                 # Pangkal ke kelingking
]

# 4. Pengaturan Buffer dan Kamera
sequence_buffer = []
sequence_length = 30
cap = cv2.VideoCapture(0)

print("Kamera terbuka. Tekan 'q' pada jendela video untuk keluar.")

predicted_label = "Menunggu Gerakan..."
confidence = 0.0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Gagal membaca kamera.")
        break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    results = detector.detect(mp_image)

    left_hand = np.zeros(63, dtype=np.float32)
    right_hand = np.zeros(63, dtype=np.float32)
    
    h, w, _ = frame.shape

    if results.hand_landmarks and len(results.hand_landmarks) > 0:
        # Urutkan tangan berdasarkan posisi X di layar (kiri ke kanan)
        sorted_hands = sorted(results.hand_landmarks, key=lambda hl: hl[0].x)

        for i, hand_landmarks in enumerate(sorted_hands):
            # Gambar garis tulang jari (hijau)
            for connection in HAND_CONNECTIONS:
                pt1 = hand_landmarks[connection[0]]
                pt2 = hand_landmarks[connection[1]]
                x1, y1 = int(pt1.x * w), int(pt1.y * h)
                x2, y2 = int(pt2.x * w), int(pt2.y * h)
                cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Gambar titik sendi (merah) dan kumpulkan koordinat
            landmarks = []
            for lm in hand_landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
                landmarks.extend([lm.x, lm.y, lm.z])
            
            landmarks_array = np.array(landmarks, dtype=np.float32)

            # Penentuan Kiri/Kanan berdasarkan posisi fisik di layar
            if len(sorted_hands) == 2:
                if i == 0:
                    left_hand = landmarks_array  # Tangan kiri layar
                else:
                    right_hand = landmarks_array # Tangan kanan layar
            else:
                if sorted_hands[0][0].x < 0.5:
                    left_hand = landmarks_array
                else:
                    right_hand = landmarks_array

    # Gabungkan koordinat (126 fitur per frame)
    frame_features = np.concatenate([left_hand, right_hand])

    # Masukkan fitur frame ke buffer sekuens
    sequence_buffer.append(frame_features)

    # Batasi buffer maksimal 30 frame terakhir
    if len(sequence_buffer) > sequence_length:
        sequence_buffer.pop(0)

    # Jalankan prediksi jika buffer sudah terkumpul 30 frame
    if len(sequence_buffer) == sequence_length:
        input_data = np.expand_dims(np.array(sequence_buffer, dtype=np.float32), axis=0)
        predictions = model.predict(input_data, verbose=0)[0]
        max_idx = np.argmax(predictions)
        confidence = predictions[max_idx]

        if confidence > 0.3:
            predicted_label = label_map_rev.get(max_idx, f"Kelas {max_idx}")
        else:
            predicted_label = "Tidak Terdeteksi"

    # Tampilan antarmuka visual
    cv2.rectangle(frame, (0, 0), (640, 50), (245, 117, 16), -1)
    text_display = f"Prediksi BISINDO: {predicted_label} ({confidence*100:.1f}%)"
    cv2.putText(frame, text_display, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    cv2.imshow('Pengujian Real-Time BISINDO', frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Memuat model LSTM...
Kamera terbuka. Tekan 'q' pada jendela video untuk keluar.
